# TRIBE v2 — Colab Backend
**Run Cell 1 → runtime auto-restarts → then run Cells 2–8 top to bottom.**

> Tip: use a **GPU runtime** (`Runtime → Change runtime type → T4 GPU`) for best results.

In [ ]:
# CELL 1 — Install everything, then auto-restart to flush old numpy from memory
!pip install -q uv
!uv pip install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!uv pip install --system fastapi "uvicorn[standard]" yt-dlp pydantic nest_asyncio huggingface_hub
!apt-get install -y -q ffmpeg 2>/dev/null || true

print("\n✓ Installs done. Auto-restarting to flush old numpy from memory...")
import os, time; time.sleep(2)
os.kill(os.getpid(), 9)   # Colab auto-restarts — this is expected

In [ ]:
# CELL 2 — Verify imports (after restart)
from tribev2.demo_utils import TribeModel
import tribev2, numpy as np, torch
print(f"tribev2 : {tribev2.__file__}")
print(f"numpy   : {np.__version__}")
print(f"torch   : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU mode)'}")
print("TribeModel ✓")

In [ ]:
# CELL 3 — Fix whisperx compute_type for CPU runtimes
# tribev2 calls whisperx as a CLI subprocess with --compute_type float16
# float16 is only supported on CUDA — this cell creates a wrapper that
# swaps it for int8 when no GPU is present.
import os, stat, subprocess, sys, torch

compute_type = "float16" if torch.cuda.is_available() else "int8"
print(f"compute_type → {compute_type}")

# Find the real whisperx binary installed by uv
r = subprocess.run(["which", "whisperx"], capture_output=True, text=True)
real_wx = r.stdout.strip()
if not real_wx:
    r2 = subprocess.run([sys.executable, "-c",
        "import whisperx, os; print(os.path.join(os.path.dirname(whisperx.__file__), '../../../bin/whisperx'))"],
        capture_output=True, text=True)
    real_wx = r2.stdout.strip()
print(f"real whisperx: {real_wx}")

# Write a wrapper at /usr/local/bin/whisperx (higher priority in PATH)
wrapper = f"""#!/bin/bash
# Auto-generated wrapper: replaces --compute_type with {compute_type}
args=()
skip=0
for arg in "$@"; do
    if [ $skip -eq 1 ]; then skip=0; continue; fi
    if [ "$arg" = "--compute_type" ]; then skip=1; continue; fi
    args+=("$arg")
done
exec "{real_wx}" "${{args[@]}}" --compute_type {compute_type}
"""

with open("/usr/local/bin/whisperx", "w") as f:
    f.write(wrapper)
os.chmod("/usr/local/bin/whisperx",
    stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)

# Verify wrapper works
r = subprocess.run(["whisperx", "--help"], capture_output=True, text=True)
print("whisperx wrapper ✓" if r.returncode == 0 else f"wrapper check: {r.stderr[:200]}")

In [ ]:
# CELL 4 — Clone / update brain-neuro
import os
repo = '/content/brain-neuro'
if os.path.exists(repo):
    !git -C {repo} pull -q
    print('brain-neuro updated ✓')
else:
    !git clone -q https://github.com/Sambhram1/brain-neuro.git {repo}
    print('brain-neuro cloned ✓')
os.chdir(repo)
print(f'cwd: {os.getcwd()}')

In [ ]:
# CELL 5 — HuggingFace login (weights download on first /analyze call)
import huggingface_hub
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN from Colab secrets ✓')
except Exception:
    hf_token = input('Enter HuggingFace token (huggingface.co/settings/tokens): ')
huggingface_hub.login(token=hf_token, add_to_git_credential=False)
print('Logged in ✓')

In [ ]:
# CELL 6 (Optional) — cookies.txt for Instagram/TikTok. Skip for YouTube.
import shutil, os
try:
    from google.colab import files
    print('Upload cookies.txt or skip this cell')
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, 'cookies.txt')
        print('cookies.txt saved ✓')
except ImportError:
    print('cookies.txt found ✓' if os.path.exists('cookies.txt') else 'No cookies.txt — YouTube only')

In [ ]:
# CELL 7 — Install cloudflared tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared ✓')

In [ ]:
# CELL 8 — Start FastAPI backend + public tunnel
import nest_asyncio, uvicorn, threading, subprocess, time, re, os
os.chdir('/content/brain-neuro')
nest_asyncio.apply()

def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)
print('FastAPI running on :8000 ✓')

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('Waiting for tunnel...')
for line in proc.stdout:
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line.decode())
    if match:
        url = match.group()
        print(f'\n{"═"*52}')
        print(f'  BACKEND URL: {url}')
        print(f'{"═"*52}')
        print('→ Paste into the frontend Backend field')
        break